In [ ]:
# 기본 모듈
import numpy as numpy
import pandas as pd
import matplotlib.pyplot as plt

# 전처리
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

# Model
from sklearn import linear_model  #Regression
from sklearn.svm import SVC  #SVM
from sklearn.ensemble import RandomForestClassifier  #RandomForest
from xgboost import XGBClassifier  #XGBoost

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Flatten, Dense
from tensorflow.keras.layers import Dropout,BatchNormalization
from tensorflow.keras.layers import Activation
from tensorflow.keras.layers import Conv2D, MaxPooling2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# 평가
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score


In [ ]:
# 1. 데이터 로딩
df = pd.read_csv('/content/drive/MyDrive/KDThome/fashion-mnist.csv/fashion-mnist_train.csv')
df.shape

(60000, 785)

In [ ]:
# 이미지라 결측치, 이상치는 없고

In [ ]:
# 학습 분리
x_data = df.drop('label', axis=1, inplace=False).values
t_data = df['label'].values

In [ ]:
# 학습데이터와 평가용 데이터 분리
x_data_train, x_data_test, t_data_train, t_data_test =\
train_test_split(x_data,
                 t_data,
                 test_size=0.3,
                 stratify=t_data,
                 random_state=42)

In [ ]:
# 정규화 진행 -> x_data_train을 기준으로 정규화
scaler = MinMaxScaler()
scaler.fit(x_data_train)
x_data_train_norm = scaler.transform(x_data_train)
x_data_test_norm = scaler.transform(x_data_test)

In [ ]:
from sklearn.metrics import accuracy_score

# sklearn - Regression 모델
skl_reg_model = linear_model.LogisticRegression()
skl_reg_model.fit(x_data_train_norm,
                  t_data_train)

# 평가
skl_reg_predict = skl_reg_model.predict(x_data_test_norm)
skl_reg_result = accuracy_score(t_data_test,
                            skl_reg_predict)
print(f'★ sklearn - Regression 모델 정확도 ★ {skl_reg_result}')

★ sklearn - Regression 모델 정확도 ★ 0.8540555555555556


/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [ ]:
# sklearn - SVM 모델
skl_svm_model = SVC(C=1,
                    kernel='rbf')
skl_svm_model.fit(x_data_train_norm,
                  t_data_train)

# 평가
skl_svm_predict = skl_svm_model.predict(x_data_test_norm)
skl_svm_result = accuracy_score(t_data_test,
                            skl_svm_predict)
print(f'★ sklearn - SVM 모델 정확도 ★ {skl_svm_result}')

★ sklearn - SVM 모델 정확도 ★ 0.887


In [ ]:
# sklearn - RandomForest 모델
skl_rf_model = RandomForestClassifier(n_estimators=50,
                                      max_depth=3,
                                      random_state=20)
skl_rf_model.fit(x_data_train_norm,
                  t_data_train)

# 평가
skl_rf_predict = skl_rf_model.predict(x_data_test_norm)
skl_rf_result = accuracy_score(t_data_test,
                            skl_rf_predict)
print(f'★ sklearn - RandomForest 모델 정확도 ★ {skl_rf_result}')

★ sklearn - RandomForest 모델 정확도 ★ 0.6846111111111111


In [ ]:
pip install optuna

In [ ]:
# XGBoost 모델
import optuna
import xgboost as xgb

# 목적 함수 정의
def objective(trial):
    params = {
        'eval_metric': 'mlogloss',
        'booster': trial.suggest_categorical('booster', ['gbtree', 'dart']),
        'lambda': trial.suggest_float('lambda', 1e-8, 10.0, log=True),
        'alpha': trial.suggest_float('alpha', 1e-8, 10.0, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'learning_rate': trial.suggest_float('learning_rate', 0.03, 0.3),
        'max_depth': trial.suggest_int('max_depth', 3, 5)
    }

    model = XGBClassifier(**params,
                              random_state=42)
    model.fit(x_data_train_norm,
              t_data_train,
              eval_set=[(x_data_test_norm, t_data_test)],
              verbose=False)

    preds = model.predict(x_data_test_norm)
    acc = accuracy_score(t_data_test, preds)
    return acc

# 스터디 생성 및 실행
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

# 최적 파라미터 출력
print("Best trial:")
print(study.best_trial.params)

best_params = study.best_trial.params
xgb_model = XGBClassifier(**best_params)
xgb_model.fit(x_data_train_norm,
              t_data_train)

# 평가
xgb_predict = xgb_model.predict(x_data_test_norm)
xgb_result = accuracy_score(t_data_test,
                            xgb_predict)
print(f'★ sklearn - XGBoost 모델 정확도 ★ {xgb_result}')

[I 2025-04-22 12:41:04,657] A new study created in memory with name: no-name-0332df89-67c5-4d75-9fe6-47679e164101


In [ ]:
# Tensorflow Keras - Regression 구현

In [ ]:
# Tensorflow Keras - DNN 구현

In [ ]:
# Tensorflow Keras - CNN 구현

cnn_model = Sequential()

cnn_model.add(Conv2D(filter=32,
                     kernel_size=(3,3),
                     activation='relu',
                     strides=(1,1),
                     input_shape=(28,28,1)))

cnn_model.add(Conv2D(filters=64,
                 kernel_size=(3,3),
                 activation='relu',
                 strides=(1,1)))

cnn_model.add(MaxPooling2D(pool_size=(2,2)))

cnn_model.add(Conv2D(filters=256,
                 kernel_size=(3,3),
                 activation='relu',
                 strides=(1,1)))

cnn_model.add(MaxPooling2D(pool_size=(2,2)))

cnn_model.add(Flatten())

cnn_model.add(Dropout(rate=0.5))

cnn_model.add(Dense(units=256,
                kernel_regularizer=l2(l2=0.1)))
cnn_model.add(BatchNormalization())
cnn_model.add(Activation('relu'))

cnn_model.add(Dense(units=10,
                activation='softmax'))

model.compile(optimizer=Adam(learning_rate=1e-4),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

es = EarlyStopping(monitor='val_loss',
                   patience=5,
                   verbose=1)

cp = ModelCheckpoint(filepath='/content/drive/MyDrive/KDThome/digit-recognizer/mnist.weights.h5',
                     save_best_only=True,
                     save_weights_only=True,
                     verbose=1,
                     monitor='val_loss')

history = model.fit(x_data_train_norm.reshape(-1,28,28,1),   # reshape(-1,28,28,1) 4차원으로 만들어줘야됨
                    t_data_train,
                    epochs=100,
                    validation_split=0.2,
                    verbose=1,
                    batch_size=100,
                    callbacks=[es,cp])